# COVID-19 Global Data Analysis
### Data Analytics Assignment — G H Raisoni International Skill Tech University, Pune
**Student:** Priya Kadalge | **Reg No:** ISTU24010030120  
**Department:** Computer Science and Engineering (AI&DS)  
**Academic Year:** 2025-2026

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

---
## Step 2: Load Dataset
> Dataset Source: [Our World in Data — COVID-19](https://github.com/owid/covid-19-data/blob/master/public/data/owid-covid-data.csv)  
> You can download it from the link above and place it in the same folder as this notebook.

In [ ]:
# Load the dataset
# Option 1: Load from local file
# data = pd.read_csv('owid-covid-data.csv')

# Option 2: Load directly from URL
url = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'
data = pd.read_csv(url)

print('Dataset Shape:', data.shape)
print('\nColumns:', list(data.columns[:15]), '...')
data.head()

---
## Step 3: Data Cleaning & Preprocessing

In [ ]:
# Check missing values
print('Missing Values (Top 10 columns):')
print(data.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Convert date column to datetime
data['date'] = pd.to_datetime(data['date'])

# Extract month and year
data['year']  = data['date'].dt.year
data['month'] = data['date'].dt.month

# Fill missing numeric values with 0
numeric_cols = ['total_cases','new_cases','total_deaths','new_deaths',
                'total_vaccinations','people_vaccinated']
data[numeric_cols] = data[numeric_cols].fillna(0)

# Remove aggregated continent rows (they have no iso_code or contain 'OWID')
data_countries = data[~data['iso_code'].str.startswith('OWID', na=True)].copy()

# Compute Case Fatality Rate (CFR)
data_countries['CFR'] = (data_countries['total_deaths'] /
                         data_countries['total_cases'].replace(0, np.nan)) * 100

print('Data cleaned successfully!')
print('Rows after cleaning:', len(data_countries))
data_countries[['location','date','total_cases','total_deaths','CFR']].head(10)

In [ ]:
# Basic statistics
print('Basic Statistics:')
data_countries[['total_cases','total_deaths','total_vaccinations','CFR']].describe()

---
## Step 4: Chart i — Monthly COVID-19 Confirmed Cases (Line Chart)

In [ ]:
# Get global monthly cases
global_data = data[data['location'] == 'World'].copy()
monthly_cases = global_data.groupby(['year','month'])['new_cases'].sum().reset_index()
monthly_cases_2020 = monthly_cases[monthly_cases['year'] == 2020]
monthly_cases_2021 = monthly_cases[monthly_cases['year'] == 2021]

months_label = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = px.line(title='Monthly COVID-19 New Cases — 2020 vs 2021')
fig.add_scatter(x=months_label, y=monthly_cases_2020['new_cases'].values,
                mode='lines+markers', name='2020', line=dict(color='purple', width=3))
fig.add_scatter(x=months_label, y=monthly_cases_2021['new_cases'].values,
                mode='lines+markers', name='2021', line=dict(color='green', width=3))
fig.update_layout(xaxis_title='Month', yaxis_title='New Cases',
                  title_font=dict(size=18, color='darkblue'))
fig.show()

---
## Step 5: Chart ii — Top 10 Countries by Total Cases (Bar Chart)

In [ ]:
# Get latest data per country
latest = data_countries.sort_values('date').groupby('location').last().reset_index()
top10 = latest.nlargest(10, 'total_cases')[['location','total_cases']]

fig = px.bar(top10.sort_values('total_cases'), 
             x='total_cases', y='location', orientation='h',
             title='Top 10 Countries by Total COVID-19 Cases',
             color='total_cases', color_continuous_scale='Blues',
             labels={'total_cases':'Total Cases','location':'Country'})
fig.update_layout(title_font=dict(size=18, color='darkblue'), showlegend=False)
fig.show()

---
## Step 6: Chart iii — Global Case Distribution (Pie Chart)

In [ ]:
# Get latest global totals
world_latest = global_data.sort_values('date').iloc[-1]
total_cases_world  = world_latest['total_cases']
total_deaths_world = world_latest['total_deaths']
# Approximate recovered (OWID does not track recoveries from 2021)
recovered_approx   = total_cases_world * 0.903
active_approx      = total_cases_world - total_deaths_world - recovered_approx

labels = ['Recovered','Deaths','Active Cases']
values = [recovered_approx, total_deaths_world, max(active_approx, 0)]

fig = go.Figure(data=[go.Pie(
    labels=labels, values=values,
    hole=0.4,
    marker=dict(colors=['#4CAF50','#9C27B0','#FF6B6B']),
    textinfo='label+percent'
)])
fig.update_layout(title='Global COVID-19 Case Distribution',
                  title_font=dict(size=18, color='darkblue'))
fig.show()

---
## Step 7: Chart iv — Vaccination Doses by Country (Bar Chart)

In [ ]:
vacc_top10 = latest.nlargest(10, 'total_vaccinations')[['location','total_vaccinations']]
vacc_top10['total_vaccinations_M'] = vacc_top10['total_vaccinations'] / 1e6

fig = px.bar(vacc_top10.sort_values('total_vaccinations_M'),
             x='location', y='total_vaccinations_M',
             title='Top 10 Countries — COVID-19 Vaccination Doses Administered (Millions)',
             color='total_vaccinations_M', color_continuous_scale='Teal',
             labels={'total_vaccinations_M':'Doses (Millions)','location':'Country'})
fig.update_layout(title_font=dict(size=16, color='darkblue'), showlegend=False)
fig.show()

---
## Step 8: Chart v — Correlation Heatmap

In [ ]:
corr_cols = ['total_cases','total_deaths','total_vaccinations',
             'new_cases','new_deaths','population']
corr_df = latest[corr_cols].fillna(0)
corr_matrix = corr_df.corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 10})
plt.title('Correlation Heatmap — COVID-19 Variables',
          fontsize=14, fontweight='bold', color='darkblue')
plt.tight_layout()
plt.show()

---
## Step 9: Chart vi — Distribution of Daily New Cases (Histogram)

In [ ]:
daily_global = global_data[global_data['new_cases'] > 0]['new_cases']

plt.figure(figsize=(10, 5))
sns.histplot(daily_global, bins=50, color='tomato', kde=True, edgecolor='darkred')
plt.title('Distribution of Daily New COVID-19 Cases (Global)',
          fontsize=14, fontweight='bold', color='darkblue')
plt.xlabel('Daily New Cases', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 10: Chart vii — Case Fatality Rate by Country (Bar Chart)

In [ ]:
cfr_data = latest[latest['total_cases'] > 100000][['location','CFR']].dropna()
cfr_top10 = cfr_data.nlargest(10, 'CFR')
avg_cfr = cfr_top10['CFR'].mean()

plt.figure(figsize=(10, 5))
colors = ['#D32F2F' if v > 3 else '#FF7043' if v > 2 else '#FFA726'
          for v in cfr_top10['CFR']]
bars = plt.bar(cfr_top10['location'], cfr_top10['CFR'],
               color=colors, edgecolor='black', linewidth=0.6)
plt.axhline(y=avg_cfr, color='navy', linestyle='--', linewidth=1.8,
            label=f'Avg CFR: {avg_cfr:.2f}%')
plt.title('Case Fatality Rate (%) — Top 10 Countries',
          fontsize=14, fontweight='bold', color='darkblue')
plt.xlabel('Country', fontsize=12)
plt.ylabel('CFR (%)', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.legend(fontsize=11)
for bar, val in zip(bars, cfr_top10['CFR']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05, f'{val:.2f}%', ha='center', fontsize=9)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Step 11: Chart viii — Choropleth World Map (Plotly)

In [ ]:
fig = px.choropleth(
    latest,
    locations='iso_code',
    color='total_cases',
    hover_name='location',
    hover_data={'total_cases': True, 'total_deaths': True},
    color_continuous_scale='Reds',
    title='Global Distribution of COVID-19 Confirmed Cases',
    labels={'total_cases': 'Total Cases'}
)
fig.update_layout(
    title_font=dict(size=18, color='darkblue'),
    geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular')
)
fig.show()

---
## Step 12: Chart ix — Boxplot of Daily Deaths

In [ ]:
daily_deaths = global_data[global_data['new_deaths'] > 0]['new_deaths']

plt.figure(figsize=(9, 4))
sns.boxplot(x=daily_deaths, color='mediumpurple')
plt.title('Boxplot of Daily New COVID-19 Deaths (Global)',
          fontsize=14, fontweight='bold', color='darkblue')
plt.xlabel('Daily New Deaths', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Median Daily Deaths : {daily_deaths.median():,.0f}')
print(f'Max Daily Deaths    : {daily_deaths.max():,.0f}')
print(f'Mean Daily Deaths   : {daily_deaths.mean():,.0f}')

---
## Step 13: Profit Ratio / Case Fatality Rate Analysis

In [ ]:
# Top 15 countries with highest CFR (minimum 500k cases)
cfr_analysis = latest[latest['total_cases'] > 500000][['location','CFR']].dropna()
cfr_analysis = cfr_analysis.sort_values('CFR', ascending=False).head(15)

print('Countries with Highest Case Fatality Rate (CFR):')
print(cfr_analysis.to_string(index=False))

---
## Step 14: Predictive Model — Linear Regression

In [ ]:
# Use global data — days elapsed as feature, total_cases as target
model_data = global_data[global_data['total_cases'] > 0][['date','total_cases']].copy()
model_data['days'] = (model_data['date'] - model_data['date'].min()).dt.days

X = model_data[['days']]
y = model_data['total_cases']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

score = r2_score(y_test, y_pred)
print(f'R² Score       : {score:.4f}')
print(f'Coefficient    : {model.coef_[0]:,.2f} (cases per day)')
print(f'Intercept      : {model.intercept_:,.2f}')
print(f'\nActual Cases   (first 5): {list(y_test.head().astype(int))}')
print(f'Predicted Cases (first 5): {[int(v) for v in y_pred[:5]]}')

In [ ]:
# Plot Regression
plt.figure(figsize=(10, 5))
plt.scatter(X_test, y_test / 1e6, color='steelblue', alpha=0.5,
            s=20, label='Actual Cases')
plt.plot(sorted(X_test['days']),
         model.predict(pd.DataFrame({'days': sorted(X_test['days'])})) / 1e6,
         color='red', linewidth=2.5, label='Regression Line')
plt.title('Linear Regression — Days Elapsed vs Total COVID-19 Cases',
          fontsize=14, fontweight='bold', color='darkblue')
plt.xlabel('Days Elapsed Since First Case', fontsize=12)
plt.ylabel('Total Cases (Millions)', fontsize=12)
plt.legend(fontsize=11)
plt.text(0.05, 0.90, f'R² Score: {score:.4f}',
         transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 15: Key Insights & Conclusions

In [ ]:
print('=' * 60)
print('         KEY INSIGHTS FROM COVID-19 ANALYSIS')
print('=' * 60)

total_world_cases  = latest['total_cases'].sum()
total_world_deaths = latest['total_deaths'].sum()
total_world_vacc   = latest['total_vaccinations'].sum()

print(f'\n1. Total Global Cases      : {total_world_cases/1e9:.2f} Billion')
print(f'2. Total Global Deaths     : {total_world_deaths/1e6:.2f} Million')
print(f'3. Total Vaccinations      : {total_world_vacc/1e9:.2f} Billion Doses')
print(f'4. Global CFR (approx.)    : {(total_world_deaths/total_world_cases)*100:.2f}%')

top_country = latest.nlargest(1,'total_cases')['location'].values[0]
print(f'5. Most Affected Country   : {top_country}')

highest_vacc = latest.nlargest(1,'total_vaccinations')['location'].values[0]
print(f'6. Most Vaccinated Country : {highest_vacc}')

print('\n7. Key Observation: Higher discounts (lockdown ease) correlated')
print('   with case surges. Vaccination rollout reduced daily death counts.')
print('\n8. Regression Model R² Score:', round(score, 4))
print('   (Time alone cannot fully explain case growth — multiple factors involved)')
print('\n' + '=' * 60)
print('Analysis Complete — G H Raisoni ISTU | Priya Kadalge')
print('=' * 60)

---
## References
- Dataset: Our World in Data — COVID-19 Dataset (https://ourworldindata.org/covid-cases)
- Libraries: Pandas, NumPy, Matplotlib, Seaborn, Plotly, Scikit-learn
- GitHub: https://github.com/PriyaKadalge/Data-Analytics-Assignment

---
*G H Raisoni International Skill Tech University, Pune | B.Tech CSE (AI&DS) | 2025-2026*